# Sprint 1: Supervised Prediction

## Brief
- We were asked to consider a binary classification problem or simple regression problem, from which we chose a probabilistic binary classification problem.
- Find a dataset and clean it.
- Choose a model to design and then test on left-out test data.
- Massage our data accordingly.
- Compare models using a metric and proper scoring rule.
- Create a comprehensive report logically ordering our work.


## Our Project
We decided to frame Expected Goals ($xG$) as a probabilistic binary classification task, evaluating model architectures on their ability to predict well calibrated continuous shot probabilities using proper scoring rules. We then use the best performing model to examine spatial patterns in shot quality and investigate how expected goals relate to actual scoring outcomes.

### xG Background
To formalise the prediction problem, let the binary outcome variable for individual shot $i$ be defined as:
$$
Y_i =
\begin{cases}
1, & \text{if shot }i\text{ results in a goal},\\
0, & \text{otherwise}.
\end{cases}
$$

We define $xG$ as the conditional probability of a goal given the available feature set:
$$
xG_i = P(Y_i=1\mid X_i),
$$
where $X_i$ represents the vector of available information regarding the shot, such as spatial coordinates (location, distance, angle) and situational context (e.g., shot type, game state). Hence xG measures the quality of a shot by looking at the events just before the ball is struck.

This formulation establishes that $xG$ is not a subjective metric of shot quality, but rather an objective probability estimated from historical observations. It is vital to distinguish between individual and aggregate interpretations:
* A single shot with an $xG$ of $0.3$ possesses a $30\%$ estimated probability of resulting in a goal.
* A collection of $100$ independent shots, each carrying an $xG$ of $0.3$, would be expected to yield approximately $30$ goals in total.

We each chose a model to design: Luke chose random forests, Dylan chose a neural network, Don chose a linear regression and Pietro chose gradient boosting. Individually we analysed our models describing their flaws, comparing them to the gold standard xG model designed by the people from whomst we took the data. Then we had each model output it's predicted xG from which we calculated a Brier Score and a log-loss, in order to consistently evaluate them.

### Data framework
The dataset was obtained via Understat, representing individual football shots across multiple European Premier League seasons. The response variable is binary ($Y_i$), indicating whether a shot resulted in a goal, and the explanatory variables ($X_i$) capture comprehensive spatial coordinates and match contexts.

Rather than allocating observations randomly, we implemented a rigorous chronological split across seasons:

| Dataset Split | Seasons Covered | Purpose |
| :--- | :--- | :--- |
| **Training** | 2021/22 -- 2023/24 | Fit model parameters |
| **Validation** | 2024/25 | Model selection and tuning |
| **Testing** | 2025/26 | Final out-of-sample evaluation |

We used a chronological split rather than random cross-validation to better reflect the real world application of an $xG$ model. Predictions must be made using historical information to estimate future outcomes. Reserving the entire 2025/26 season as an unseen test period ensures an honest, out-of-sample evaluation free from data leakage. Note that we also had the 2026/27 season's shot data too but since that was a incomplete dataset with insufficient number of shots, we decided to not use it even for testing.

### Performance metrics

1. The **Brier score** measures the mean squared difference between predicted probabilities ($f_i$) and observed outcomes ($y_i$):
$$BS = \frac{1}{N}\sum_{t=1}^{N}{(f_t-\sigma_t)^2};$$
It is bounded between 0 and 1 for binary outcomes (lower values indicate better calibration). It is a strictly proper scoring rule, meaning a model minimises its expected score by reporting its true estimated probabilities.
This scoring system penalises overconfidence as it calculates probabilistic distance between prediction and actual occurence, but is capped at a penalty of less than $1$, as an overconfident mistake; $f_t = 0.001$ and an outcome $\sigma_t = 1$ gives a penalty of $0.999^2$.

2. The **log loss** idea is similar and is calculated by:
$$LogLoss = -\frac{1}{N}\sum_{i=1}^N\bigg{[}\sigma_tln({f_t})+(1-\sigma_t)ln(1-{f_t})\bigg{]}.$$
This measures the information-theoretic entropy, and assuming that a goal occurs ($\sigma_t = 1$) then the loss is $L = -\log{f_t}$. This means the loss is unbounded since the gradient $\frac{\partial L}{\partial f_t}=\frac{-1}{f_t}$, so as $f_t\rightarrow0^{+}$ the gradient approaches $+\infty$.

3. The **ROC-AUC** measures a model's capacity to discriminate between goals and non-goals across all classification thresholds ($0.5$ for random ranking up to $1.0$ for perfect discrimination). While Brier score and log loss assess probability calibration, ROC-AUC evaluates ranking performance.

As a group we comparatively analysed our models basing the Brier Scores and log-loss on the held-out dataset which was the 2025/26 season shots. Using these complementary metrics provides a holistic evaluation: Brier score measures overall squared error, log loss penalises severe overconfidence and ROC-AUC evaluates discrimination independent of probability scaling. We used these proper scoring rules as we do not want to categorise shots into binary goal or no-goal classes but instead as continuous conditional probabilities.

### Accuracy

Standard classification accuracy is deeply misleading for probabilistic predictions due to severe class imbalance. 

Across our training dataset, historical goal frequency is approximately $10\%$. If we evaluate this using classification accuracy with a standard threshold of $p\geq0.5$ where:

$$\text{Accuracy} = \frac{\text{True Positives}+\text{True Negatives}}{\text{Total Predictions}}$$

Then for a dummy model that predicts $f_t=0$ for every shot, it has an accuracy of:

$$\frac{0+0.9N}{N}=90\%$$

Despite this model having a perceived 90% accuracy, it has evidently no predictive utility. Brier Score and log loss solve this issue by heavily penalising naive baseline strategies especially confident incorrect predictions, which giving every goal an $xG$ of 0.


We evaluate our models against the existing $xG$ estimates provided by Understat. Rather than treating Understat as an objectively optimal gold standard, we use it as an established external benchmark to test whether our independently trained models can achieve comparable or superior predictive performance on unseen data. 

Finally, to answer our question we select the model with the highest benchmarks to obtain an xG value for each shot throughout our test season. Using these values, we designed heat maps. We uniformly sectioned off both the attacking and defending side of the pitch for every team in order to visualise spatially which areas of the pitch teams are performing poorly or well. We did this for both the shots they made and those they conceded in order to get a more rounded breakdown of a teams weakpoints. The heatmaps show what area of the pitch a team may be over/underperforming compared to their xG by taking the difference between actual goals conceded in that area of the pitch and the amount our xG model predicted them to score. If they score more than their predicted xG the square is green and red if they scored less.

We chose this since visual spatial heatmaps convert continuous probability residuals ($\text{Actual Goals} - \text{Predicted} xG$) into a more intuitive representation. This exposes localised bias that aggregated seasonal totals may have masked. This allowed us to pinpoint more specific pitch areas where model predictions diverged from reality, which directly distinguishes between structural model limitations such as unobserved defender density in crowded zones and tactical patterns.

## Team Contributions
- **Don**: Cleaned and massaged data pipelines, designed the baseline linear regression model, managed repository formatting and overviewed the final report structure.
- **Pietro**: Designed the gradient boosting model and produced the introductory material.
- **Dylan**: Evaluated dataset suitability, imported data feeds, designed the neural network model in R and assured all notebooks ran smoothly
- **Luke**: Built the random forest model in Python, assembled comparative model evaluations and generated spatial shot heatmaps.